# MTUS vs Simulation Time-Use Comparison

Compare France 2009 MTUS daily time-use aggregates with the latest Greater Paris synthetic micro-activity simulation output.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "00_ile_de_france_exploration",
]
NOTEBOOK_DIR = next(
    (path for path in NOTEBOOK_DIR_CANDIDATES if (path / "02_mtus_exploration.ipynb").exists()),
    Path.cwd(),
)
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "00_ile_de_france_exploration" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MTUS_PATH = NOTEBOOK_DIR / "../../data/mtus/MTUS_haf.dta"
SIM_RESULTS_DIR = PROJECT_ROOT / "data" / "gparis" / "results"
SIM_ACTIVITY_CANDIDATES = sorted(SIM_RESULTS_DIR.glob("gparis_simulation_core_trajectories_*_activities.parquet"))
assert SIM_ACTIVITY_CANDIDATES, f"No simulation activity parquet files found in {SIM_RESULTS_DIR}."
SIM_ACTIVITIES_PATH = SIM_ACTIVITY_CANDIDATES[-1]

MTUS_COUNTRY = "France"
MTUS_SURVEY = 2009
WEIGHT_COL = "propwt"
GROUP_ORDER = ["All", "Weekday", "Weekend"]

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"MTUS input: {MTUS_PATH.resolve()}")
print(f"Simulation activities input: {SIM_ACTIVITIES_PATH.resolve()}")

Project root: /home/gustavo/citybehavex
MTUS input: /home/gustavo/citybehavex/data/mtus/MTUS_haf.dta
Simulation activities input: /home/gustavo/citybehavex/data/gparis/results/gparis_simulation_core_trajectories_20260706T151504_activities.parquet


## Shared Category Mapping

The simulation micro-activities and MTUS HAF columns are not one-to-one, so this notebook compares both sources through an explicit common category layer.

In [2]:
from citybehavex.activities.catalog import build_catalog
from citybehavex.reports.comparison import _micro_activity_daily_usage_data

COMMON_CATEGORY_ORDER = [
    "sleep",
    "personal_care",
    "work_study",
    "household_care",
    "shopping_errands_health",
    "leisure_social_sport",
    "travel_missing",
]

MTUS_CATEGORY_MAP = {
    "sleep": ["sleep"],
    "personal_care": ["eatdrink", "selfcare"],
    "work_study": ["paidwork", "educatn"],
    "household_care": ["foodprep", "cleanetc", "maintain", "garden", "petcare", "eldcare", "pkidcare", "ikidcare"],
    "shopping_errands_health": ["shopserv"],
    "leisure_social_sport": ["sportex", "tvradio", "read", "compint", "goout", "leisure", "religion", "volorgwk"],
    "travel_missing": ["commute", "travel", "missing"],
}

SIM_CATEGORY_MAP = {
    "sleep": ["sleep"],
    "personal_care": ["eatdrink", "selfcare"],
    "work_study": ["paidwork", "educatn"],
    "household_care": ["foodprep", "cleanetc", "maintain", "garden", "petcare", "eldcare", "pkidcare", "ikidcare"],
    "shopping_errands_health": ["shopserv"],
    "leisure_social_sport": ["sportex", "tvradio", "read", "compint", "goout", "leisure", "religion", "volorgwk"],
    "travel_missing": ["commute", "travel", "missing"],
}

MTUS_TIME_COLS = sorted({col for cols in MTUS_CATEGORY_MAP.values() for col in cols})
SIM_ACTIVITY_TO_CATEGORY = {
    activity_name: category
    for category, activity_names in SIM_CATEGORY_MAP.items()
    for activity_name in activity_names
}

mapping_table = pd.DataFrame(
    {
        "category": COMMON_CATEGORY_ORDER,
        "mtus_columns": [", ".join(MTUS_CATEGORY_MAP[category]) for category in COMMON_CATEGORY_ORDER],
        "simulation_activities": [", ".join(SIM_CATEGORY_MAP[category]) or "-" for category in COMMON_CATEGORY_ORDER],
    }
)
display(mapping_table)

,category,mtus_columns,simulation_activities
0,sleep,sleep,sleep
1,personal_care,"eatdrink, selfcare","eatdrink, selfcare"
2,work_study,"paidwork, educatn","paidwork, educatn"
3,household_care,"foodprep, cleanetc, maintain, garden, petcare,...","foodprep, cleanetc, maintain, garden, petcare,..."
4,shopping_errands_health,shopserv,shopserv
5,leisure_social_sport,"sportex, tvradio, read, compint, goout, leisur...","sportex, tvradio, read, compint, goout, leisur..."
6,travel_missing,"commute, travel, missing","commute, travel, missing"


## Load and Aggregate MTUS

In [3]:
mtus_columns = ["country", "survey", "day", WEIGHT_COL] + MTUS_TIME_COLS
mtus = pd.read_stata(MTUS_PATH, columns=mtus_columns)
mtus = mtus[(mtus["country"] == MTUS_COUNTRY) & (mtus["survey"] == MTUS_SURVEY)].copy()

assert not mtus.empty, f"No MTUS rows found for {MTUS_COUNTRY} {MTUS_SURVEY}."

diary_totals = mtus[MTUS_TIME_COLS].sum(axis=1)
assert np.allclose(diary_totals, 1440), "At least one MTUS diary does not sum to 1440 minutes."

mtus["day_group"] = np.where(mtus["day"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")

print(f"MTUS rows: {len(mtus):,}")
display(mtus["day"].value_counts().rename_axis("day").reset_index(name="rows"))
display(mtus[[WEIGHT_COL]].describe())

MTUS rows: 27,903


,day,rows
0,Sunday,6599
1,Saturday,6392
2,Monday,3130
3,Wednesday,3004
4,Tuesday,2954
5,Friday,2933
6,Thursday,2891
7,non response,0
8,unspecified weekday,0
9,unspecified weekend day,0


,propwt
count,27903.000000
mean,0.983801
std,0.449481
min,0.000000
25%,0.609866
50%,0.964249
75%,1.302753
max,6.515524


In [4]:
def weighted_mean_minutes(frame: pd.DataFrame, columns: list[str], weight_col: str = WEIGHT_COL) -> float:
    weights = pd.to_numeric(frame[weight_col], errors="coerce").fillna(0).astype(float)
    values = frame[columns].sum(axis=1).astype(float)
    if weights.sum() <= 0:
        return np.nan
    return float(np.average(values, weights=weights))


def mtus_group_frame(frame: pd.DataFrame, group_name: str) -> pd.DataFrame:
    rows = []
    for category in COMMON_CATEGORY_ORDER:
        rows.append(
            {
                "group": group_name,
                "category": category,
                "mtus_minutes": weighted_mean_minutes(frame, MTUS_CATEGORY_MAP[category]),
            }
        )
    return pd.DataFrame(rows)


mtus_summary = pd.concat(
    [
        mtus_group_frame(mtus, "All"),
        mtus_group_frame(mtus[mtus["day_group"] == "Weekday"], "Weekday"),
        mtus_group_frame(mtus[mtus["day_group"] == "Weekend"], "Weekend"),
    ],
    ignore_index=True,
)

display(mtus_summary.pivot(index="category", columns="group", values="mtus_minutes").loc[COMMON_CATEGORY_ORDER, GROUP_ORDER].round(1))

group,All,Weekday,Weekend
category,,,
sleep,506.2,496.9,529.4
personal_care,178.4,170.4,198.3
work_study,165.0,210.3,51.9
household_care,178.7,174.9,188.2
shopping_errands_health,27.0,27.0,27.1
leisure_social_sport,307.5,282.1,371.0
travel_missing,77.2,78.4,74.1


## Load and Aggregate Simulation

In [5]:
catalog = build_catalog()
activity_labels = {activity.idx: activity.name for activity in catalog}

sim_raw = pd.read_parquet(SIM_ACTIVITIES_PATH)
required_sim_columns = {"uid", "activity", "arrival", "departure"}
missing_sim_columns = sorted(required_sim_columns - set(sim_raw.columns))
assert not missing_sim_columns, f"Simulation activities table missing columns: {missing_sim_columns}"

sim = sim_raw.copy()
sim["arrival"] = pd.to_datetime(sim["arrival"], errors="coerce")
sim["departure"] = pd.to_datetime(sim["departure"], errors="coerce")
sim["activity"] = pd.to_numeric(sim["activity"], errors="coerce")
sim = sim.dropna(subset=["uid", "activity", "arrival", "departure"])
sim = sim[sim["departure"] > sim["arrival"]].copy()
sim["activity"] = sim["activity"].astype(int)
sim["activity_name"] = sim["activity"].map(activity_labels)

unknown_activity_ids = sorted(set(sim[sim["activity_name"].isna()]["activity"].unique()))
assert not unknown_activity_ids, f"Unknown simulation activity ids: {unknown_activity_ids}"
assert not sim.empty, "Simulation activities table has no valid intervals."

print(f"Valid simulation intervals: {len(sim):,}")
print(f"Agents: {sim['uid'].nunique():,}")
print(f"Interval span: {sim['arrival'].min()} to {sim['departure'].max()}")
display(sim["activity_name"].value_counts().rename_axis("activity_name").reset_index(name="intervals"))

Valid simulation intervals: 73,349
Agents: 500
Interval span: 2026-07-06 00:00:00 to 2026-07-13 00:00:00


,activity_name,intervals
0,eatdrink,20141
1,selfcare,6997
2,travel,6711
3,paidwork,3833
4,read,3681
5,tvradio,3343
6,compint,3090
7,commute,2992
8,leisure,2949
9,pkidcare,2643


In [6]:
def split_intervals_at_midnight(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in frame[["uid", "activity_name", "arrival", "departure"]].itertuples(index=False):
        current = pd.Timestamp(row.arrival)
        end = pd.Timestamp(row.departure)
        while current < end:
            next_midnight = current.normalize() + pd.Timedelta(days=1)
            segment_end = min(end, next_midnight)
            minutes = (segment_end - current).total_seconds() / 60
            rows.append(
                {
                    "uid": row.uid,
                    "date": current.date(),
                    "day_group": "Weekend" if current.dayofweek >= 5 else "Weekday",
                    "activity_name": row.activity_name,
                    "minutes": minutes,
                }
            )
            current = segment_end
    return pd.DataFrame(rows)


sim_segments = split_intervals_at_midnight(sim)
sim_segments["category"] = sim_segments["activity_name"].map(SIM_ACTIVITY_TO_CATEGORY)

unmapped_sim_activities = sorted(sim_segments[sim_segments["category"].isna()]["activity_name"].dropna().unique())
assert not unmapped_sim_activities, f"Unmapped simulation activity names: {unmapped_sim_activities}"
assert not sim_segments.empty, "No simulation segments remained after midnight splitting."

daily_totals = sim_segments.groupby(["uid", "date"], as_index=False)["minutes"].sum()
print(f"Simulation agent-days: {len(daily_totals):,}")
display(daily_totals["minutes"].describe().to_frame("minutes_per_agent_day"))
display(sim_segments.groupby("day_group")["date"].nunique().rename("dates").reset_index())

Simulation agent-days: 3,500


,minutes_per_agent_day
count,3.500000e+03
mean,1.440000e+03
std,1.274866e-14
min,1.440000e+03
25%,1.440000e+03
50%,1.440000e+03
75%,1.440000e+03
max,1.440000e+03


,day_group,dates
0,Weekday,5
1,Weekend,2


In [7]:
def simulation_group_frame(segments: pd.DataFrame, group_name: str) -> pd.DataFrame:
    if group_name == "All":
        group_segments = segments.copy()
    else:
        group_segments = segments[segments["day_group"] == group_name].copy()

    agent_days = group_segments[["uid", "date"]].drop_duplicates()
    assert not agent_days.empty, f"No simulation agent-days for {group_name}."

    category_minutes = (
        group_segments.groupby(["uid", "date", "category"], as_index=False)["minutes"].sum()
    )
    complete_index = pd.MultiIndex.from_product(
        [agent_days["uid"].unique(), agent_days["date"].unique(), COMMON_CATEGORY_ORDER],
        names=["uid", "date", "category"],
    )
    complete = pd.DataFrame(index=complete_index).reset_index()
    complete = complete.merge(category_minutes, on=["uid", "date", "category"], how="left")
    complete["minutes"] = complete["minutes"].fillna(0)

    summary = complete.groupby("category", as_index=False)["minutes"].mean()
    summary = summary.rename(columns={"minutes": "simulation_minutes"})
    summary["group"] = group_name
    return summary[["group", "category", "simulation_minutes"]]


simulation_summary = pd.concat(
    [simulation_group_frame(sim_segments, group_name) for group_name in GROUP_ORDER],
    ignore_index=True,
)

display(simulation_summary.pivot(index="category", columns="group", values="simulation_minutes").loc[COMMON_CATEGORY_ORDER, GROUP_ORDER].round(1))

group,All,Weekday,Weekend
category,,,
sleep,245.8,255.3,222.0
personal_care,273.1,218.9,408.9
work_study,225.7,292.3,59.3
household_care,229.2,219.3,253.7
shopping_errands_health,3.8,4.3,2.5
leisure_social_sport,433.4,421.6,463.0
travel_missing,28.9,28.3,30.6


## MTUS vs Simulation Comparison

In [8]:
comparison = mtus_summary.merge(simulation_summary, on=["group", "category"], how="outer")
comparison["category"] = pd.Categorical(comparison["category"], categories=COMMON_CATEGORY_ORDER, ordered=True)
comparison["group"] = pd.Categorical(comparison["group"], categories=GROUP_ORDER, ordered=True)
comparison = comparison.sort_values(["group", "category"]).reset_index(drop=True)
comparison["absolute_difference_minutes"] = comparison["simulation_minutes"] - comparison["mtus_minutes"]
comparison["percent_difference"] = np.where(
    comparison["mtus_minutes"].abs() > 0,
    comparison["absolute_difference_minutes"] / comparison["mtus_minutes"] * 100,
    np.nan,
)
comparison["share_of_day_difference_pct_points"] = comparison["absolute_difference_minutes"] / 1440 * 100

assert set(comparison["group"].astype(str)) == set(GROUP_ORDER), "Missing one or more comparison groups."
assert len(comparison) == len(GROUP_ORDER) * len(COMMON_CATEGORY_ORDER), "Unexpected comparison table size."

comparison_display = comparison.copy()
numeric_cols = [
    "mtus_minutes",
    "simulation_minutes",
    "absolute_difference_minutes",
    "percent_difference",
    "share_of_day_difference_pct_points",
]
comparison_display[numeric_cols] = comparison_display[numeric_cols].round(1)
display(comparison_display)

,group,category,mtus_minutes,simulation_minutes,absolute_difference_minutes,percent_difference,share_of_day_difference_pct_points
0,All,sleep,506.2,245.8,-260.4,-51.4,-18.1
1,All,personal_care,178.4,273.1,94.8,53.1,6.6
2,All,work_study,165.0,225.7,60.7,36.8,4.2
3,All,household_care,178.7,229.2,50.5,28.2,3.5
4,All,shopping_errands_health,27.0,3.8,-23.2,-85.9,-1.6
5,All,leisure_social_sport,307.5,433.4,125.9,41.0,8.7
6,All,travel_missing,77.2,28.9,-48.3,-62.5,-3.4
7,Weekday,sleep,496.9,255.3,-241.5,-48.6,-16.8
8,Weekday,personal_care,170.4,218.9,48.4,28.4,3.4
9,Weekday,work_study,210.3,292.3,82.0,39.0,5.7


In [9]:
comparison_long = comparison.melt(
    id_vars=["group", "category"],
    value_vars=["mtus_minutes", "simulation_minutes"],
    var_name="source",
    value_name="minutes_per_day",
)
comparison_long["source"] = comparison_long["source"].map(
    {"mtus_minutes": "MTUS France 2009", "simulation_minutes": "Simulation"}
)

fig = px.bar(
    comparison_long,
    x="category",
    y="minutes_per_day",
    color="source",
    facet_col="group",
    barmode="group",
    category_orders={"category": COMMON_CATEGORY_ORDER, "group": GROUP_ORDER},
    labels={"category": "Common category", "minutes_per_day": "Mean minutes/day", "source": "Source"},
    title="Mean daily time-use: MTUS France 2009 vs Greater Paris simulation",
    height=520,
)
fig.update_xaxes(tickangle=35)
fig.show()

In [10]:
fig = px.bar(
    comparison,
    x="absolute_difference_minutes",
    y="category",
    color="absolute_difference_minutes",
    facet_col="group",
    category_orders={"category": list(reversed(COMMON_CATEGORY_ORDER)), "group": GROUP_ORDER},
    labels={"absolute_difference_minutes": "Simulation - MTUS minutes/day", "category": "Common category"},
    title="Simulation difference from MTUS daily minutes",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    height=520,
)
fig.add_vline(x=0, line_width=1, line_color="black")
fig.show()

## Synthetic Time-of-Day Context

This chart reuses the report-side micro-activity usage logic to show the simulation's within-day activity mix. MTUS HAF does not contain equivalent clock-time episode profiles, so this is context rather than a direct MTUS comparison.

In [11]:
usage = _micro_activity_daily_usage_data(sim_raw, bin_size_minutes=10)

fig = go.Figure()
for series in usage["series"]:
    fig.add_trace(
        go.Scatter(
            x=usage["x"],
            y=series["values"],
            mode="lines",
            stackgroup="micro_activity_usage",
            name=series["name"],
        )
    )

fig.update_layout(
    title="Synthetic micro-activity usage over the day",
    xaxis_title="Time of day",
    yaxis_title="Share of synthetic micro-activity time (%)",
    hovermode="x unified",
    template="plotly_white",
    legend_title_text="Micro-activity",
    height=520,
    margin=dict(l=48, r=24, t=64, b=48),
)
fig.update_xaxes(tickmode="array", tickvals=usage["x"][::12], tickangle=0)
fig.update_yaxes(range=[0, 101])
fig.show()